In [ ]:
# 01 문제정의
# 02 데이터가져오기
# 03 EDA
# 04 데이터전처리
# 05 검증데이터 분할 및 학습
# 06 테스트 및 검증지표 확인
# 07 Model 내보내기 , 테스트파일 내보내기

In [ ]:
# IMPORT

In [1]:
import pandas as pd
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
# train = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch2/train.csv')
# test = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch2/test.csv')

print('train:', train.shape, '| test:', test.shape)

train: (29304, 16) | test: (3257, 15)


In [3]:
# y값 분리
y = train['income'].map({'<=50K': 0, '>50K': 1})   # 문자 → 0/1

In [4]:
train=train.select_dtypes(include="number");
test=test.select_dtypes(include="number");
train=train.drop(columns=['fnlwgt','capital.gain','capital.loss'])
test=test.drop(columns=['fnlwgt','capital.gain','capital.loss'])

In [ ]:
# 데이터 전처리

In [5]:
num_cols = ['age', 'hours.per.week']  # 수치형 데이터

for c in num_cols:
    m = train[c].median()            # train 기준 중앙값
    train[c] = train[c].fillna(m)
    test[c] = test[c].fillna(m)

In [6]:
train = train.drop(columns=['id'])
train.shape

(29304, 3)

In [7]:
test_id = test['id']                                # 제출용 보관
test = test.drop(columns=['id'])

In [12]:
test.shape

(3257, 14)

In [12]:
train

,age,education.num,hours.per.week
0,34.0,10,40.0
1,58.0,9,40.0
2,48.0,10,38.0
3,58.0,10,40.0
4,41.0,10,54.0
...,...,...,...
29299,28.0,6,40.0
29300,44.0,16,38.0
29301,41.0,9,40.0
29302,43.0,9,40.0


In [13]:
all_df = pd.concat([train, test], axis=0)           # 합쳐서 인코딩
all_oh = pd.get_dummies(all_df)


In [15]:
all_df['marital.status'].unique()

array(['Married-civ-spouse', 'Widowed', 'Divorced', 'Never-married',
       'Married-spouse-absent', 'Separated', 'Married-AF-spouse'],
      dtype=object)

In [14]:
all_oh

,age,education.num,hours.per.week
0,34.0,10,40.0
1,58.0,9,40.0
2,48.0,10,38.0
3,58.0,10,40.0
4,41.0,10,54.0
...,...,...,...
3252,28.0,10,40.0
3253,52.0,9,40.0
3254,25.0,9,38.0
3255,36.0,9,40.0


In [15]:
X = all_oh.iloc[:len(train)]                        # 다시 train 부분
X_test = all_oh.iloc[len(train):]                   # test 부분
print('인코딩 후 컬럼 수:', X.shape[1])

인코딩 후 컬럼 수: 3


In [ ]:
# 검증 데이터 분할

In [16]:
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y)
print(X_tr.shape, X_val.shape)

(23443, 3) (5861, 3)


In [ ]:
# 데이터 학습

In [17]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, random_state=0)
rf.fit(X_tr, y_tr)

RandomForestClassifier(random_state=0)

In [ ]:
# 모델 검증 지표

In [18]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
pred = rf.predict(X_val)
print(pred)
proba = rf.predict_proba(X_val)[:, 1]   # 1(>50K)일 확률 열만 (2차원 슬라이싱)
print(proba)
print('정확도 :', round(accuracy_score(y_val, pred), 4))
print('ROC-AUC:', round(roc_auc_score(y_val, proba), 4))
# # print(classification_report(y_val, pred))

[0 0 1 ... 0 0 0]
[0.47625017 0.01133846 0.7475751  ... 0.         0.         0.        ]
정확도 : 0.7674
ROC-AUC: 0.764


In [20]:
# 8) test 예측 & 제출 파일

train.describe()

,age,education.num,hours.per.week
count,29304.000000,29304.000000,29304.000000
mean,38.552587,10.080842,40.434036
std,13.626057,2.570824,12.321306
min,-38.000000,1.000000,1.000000
25%,28.000000,9.000000,40.000000
50%,37.000000,10.000000,40.000000
75%,48.000000,12.000000,45.000000
max,90.000000,16.000000,99.000000


In [40]:
X_test

,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week,workclass_Federal-gov,workclass_Local-gov,workclass_Never-worked,workclass_Private,...,native.country_Portugal,native.country_Puerto-Rico,native.country_Scotland,native.country_South,native.country_Taiwan,native.country_Thailand,native.country_Trinadad&Tobago,native.country_United-States,native.country_Vietnam,native.country_Yugoslavia
0,39.0,114055,13,0,0,40.0,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
1,38.0,254114,10,0,0,40.0,False,False,False,True,...,False,False,False,False,False,False,False,True,False,False
2,44.0,55395,9,0,0,40.0,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
3,47.0,28035,13,0,0,50.0,False,False,False,True,...,False,False,False,False,False,False,False,True,False,False
4,62.0,186611,9,0,0,40.0,False,False,False,True,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3252,28.0,202558,10,0,0,40.0,False,True,False,False,...,False,False,False,False,False,False,False,True,False,False
3253,52.0,94391,9,0,0,40.0,False,False,False,True,...,False,False,False,False,False,False,False,True,False,False
3254,25.0,109526,9,0,0,38.0,False,True,False,False,...,False,False,False,False,False,False,False,True,False,False
3255,36.0,242713,9,0,0,40.0,False,False,False,True,...,False,False,False,False,False,False,False,True,False,False


In [41]:
rf.predict_proba(X_test)

array([[0.99, 0.01],
       [0.98, 0.02],
       [0.91, 0.09],
       ...,
       [0.86, 0.14],
       [0.99, 0.01],
       [0.97, 0.03]], shape=(3257, 2))

In [43]:
test_proba = rf.predict_proba(X_test)[:, 1]
submit = pd.DataFrame({'id': test_id, 'income': test_proba})
submit.to_csv('submission.csv', index=False)   # index=False 필수
submit.head(15)

,id,income
0,11574,0.01
1,15847,0.02
2,17655,0.09
3,19790,0.97
4,31812,0.04
5,7548,0.14
6,13093,0.49
7,18332,0.20
8,28474,0.27
9,4213,0.00


In [13]:
# model.pkl 파일변환

In [19]:
import joblib

joblib.dump(rf, "model.pkl")             # 2) 파일로 저장


['model.pkl']

In [21]:
import joblib
# 모델 feature , target 묶기
bundle = {
    "model": rf,
    "features": list(X_tr.columns),          # 학습에 쓴 컬럼 순서
    "targets": ["<=50K", ">50K"],               # 클래스 라벨
}
joblib.dump(bundle, "model.pkl")

['model.pkl']